# Phase 7A — Review Queue Backend

## 1. Phase Overview

Phase 7A introduced the backend review-queue workflow for the VIGILOX Document Intelligence system.

Before this phase, the system could already:

- Analyze documents
- Persist machine results
- Generate review decisions
- Store human reviews
- Preserve audit events
- Retrieve documents and review history

However, there was no centralized way for a human reviewer to retrieve all documents that still required manual review.

Phase 7A solved this by introducing a dedicated review queue.

The completed workflow is:

```text
Document Analysis
      ↓
Machine Review Decision
      ↓
REVIEW_REQUIRED
      ↓
PostgreSQL
      ↓
Review Queue Query
      ↓
GET /api/v1/reviews/queue
      ↓
Human Reviewer
      ↓
APPROVE / REJECT / CORRECT
      ↓
Human Review Persisted
      ↓
Document Removed from Pending Queue
````

---

# 2. Phase 7A Objectives

The main objectives were:

1. Retrieve documents marked `REVIEW_REQUIRED`.
2. Exclude documents marked `AUTO_ACCEPT`.
3. Exclude documents already processed by a human reviewer.
4. Support priority filtering.
5. Support document-type filtering.
6. Order the queue by review priority.
7. Return older documents first within the same priority.
8. Expose the queue through FastAPI.
9. Validate API responses using Pydantic schemas.
10. Verify the complete review workflow against PostgreSQL.

---

# 3. Review Queue Requirements

A document is considered pending review when:

```text
Machine decision = REVIEW_REQUIRED
AND
No HumanReview record exists
```

A document must not appear in the pending queue when:

```text
Machine decision = AUTO_ACCEPT
```

or when:

```text
Human review already exists
```

regardless of whether the human action was:

```text
APPROVE
REJECT
CORRECT
```

---

# 4. Queue Ordering

Review cases are ordered by operational priority:

```text
HIGH
 ↓
MEDIUM
 ↓
LOW
```

Within the same priority, older documents are returned first:

```text
created_at ASC
```

This produces queue behaviour similar to:

```text
HIGH — oldest
HIGH — newer
MEDIUM — oldest
MEDIUM — newer
LOW — oldest
LOW — newer
```

The intention is to process urgent cases first while also preventing older cases from remaining indefinitely in the queue.

---

# 5. Repository Layer

The review-queue database query was added to:

```text
src/db/repositories.py
```

A new repository was introduced:

```python
ReviewQueueRepository
```

with the main method:

```python
get_review_queue(
    priority=None,
    document_type=None,
)
```

The repository queries:

```text
documents
+
document_analyses
```

and checks:

```text
human_reviews
```

to determine whether a case is still pending.

---

# 6. JSONB Review Decision Querying

The machine review decision is stored in PostgreSQL as JSONB.

The repository extracts fields such as:

```text
review_decision["decision"]
review_decision["priority"]
```

The queue selects only:

```text
decision = REVIEW_REQUIRED
```

This allows the machine decision to remain stored as a structured JSON object while still supporting SQL-level filtering.

---

# 7. Human Review Exclusion

The queue uses an existence check against:

```text
human_reviews
```

Conceptually:

```text
IF HumanReview exists for document
    → exclude from pending queue
```

Therefore:

```text
REVIEW_REQUIRED + no human review
→ pending

REVIEW_REQUIRED + APPROVE
→ removed

REVIEW_REQUIRED + REJECT
→ removed

REVIEW_REQUIRED + CORRECT
→ removed
```

This design keeps the pending queue focused only on unresolved work.

---

# 8. Optional Filters

The repository supports two optional filters.

## Priority

Supported values:

```text
HIGH
MEDIUM
LOW
```

Example:

```text
priority = HIGH
```

returns only high-priority pending reviews.

---

## Document Type

Supported document types are:

```text
guard_license
sia_badge
id_card
```

Example:

```text
document_type = id_card
```

returns pending ID-card reviews only.

---

## Combined Filters

Both filters can be used together:

```text
priority = MEDIUM
document_type = guard_license
```

This returns only medium-priority guard licences awaiting human review.

---

# 9. Query Service

The review-queue functionality was integrated into:

```text
src/db/query_service.py
```

The `DocumentQueryService` now includes:

```python
get_review_queue(
    priority=None,
    document_type=None,
)
```

The query service acts as the layer between:

```text
Repository
    ↓
Database models
    ↓
API-ready dictionaries
```

---

# 10. Filter Normalization

The query service normalizes incoming filter values.

For example:

```text
high
→ HIGH
```

and:

```text
ID_CARD
→ id_card
```

This allows the API to remain user-friendly while maintaining consistent internal values.

---

# 11. Queue Response Structure

The query service converts SQLAlchemy models into clean response objects.

A queue item contains:

```json
{
  "document_id": "...",
  "analysis_id": "...",
  "original_filename": "guard_license.jpg",
  "content_type": "image/jpeg",
  "document_type": "guard_license",
  "processing_status": "PROCESSED",
  "review_decision": "REVIEW_REQUIRED",
  "review_priority": "MEDIUM",
  "reason_codes": [
    "DOCUMENT_EXPIRED"
  ],
  "review_issues": [],
  "anomaly_issues": [],
  "created_at": "...",
  "analysis_created_at": "..."
}
```

The complete service response is:

```json
{
  "total": 1,
  "filters": {
    "priority": null,
    "document_type": null
  },
  "documents": []
}
```

---

# 12. API Response Schemas

Review-queue Pydantic models were added to:

```text
src/api/schemas.py
```

The new models are:

```text
ReviewQueueFilters
ReviewQueueItem
ReviewQueueResponse
```

The existing:

```text
HumanReviewRequest
```

schema was preserved.

---

# 13. `ReviewQueueFilters`

This schema describes the active queue filters.

Example:

```json
{
  "priority": "MEDIUM",
  "document_type": "guard_license"
}
```

Priority is restricted to:

```text
HIGH
MEDIUM
LOW
```

or `null`.

---

# 14. `ReviewQueueItem`

Each queue item contains:

```text
document_id
analysis_id
original_filename
content_type
document_type
processing_status
review_decision
review_priority
reason_codes
review_issues
anomaly_issues
created_at
analysis_created_at
```

The schema restricts:

```text
review_decision
```

to:

```text
REVIEW_REQUIRED
```

because only pending review documents should be returned from this endpoint.

Priority is restricted to:

```text
HIGH
MEDIUM
LOW
```

---

# 15. `ReviewQueueResponse`

The complete response model contains:

```text
total
filters
documents
```

Example:

```json
{
  "total": 2,
  "filters": {
    "priority": null,
    "document_type": null
  },
  "documents": [
    {
      "document_id": "abc",
      "analysis_id": "xyz",
      "original_filename": "guard.jpg",
      "content_type": "image/jpeg",
      "document_type": "guard_license",
      "processing_status": "PROCESSED",
      "review_decision": "REVIEW_REQUIRED",
      "review_priority": "MEDIUM",
      "reason_codes": [
        "DOCUMENT_EXPIRED"
      ],
      "review_issues": [],
      "anomaly_issues": [],
      "created_at": "2026-08-19T10:00:00+00:00",
      "analysis_created_at": "2026-08-19T10:00:01+00:00"
    }
  ]
}
```

---

# 16. FastAPI Review Queue Endpoint

A new endpoint was added to:

```text
src/api/main.py
```

The endpoint is:

```http
GET /api/v1/reviews/queue
```

It uses:

```python
response_model=ReviewQueueResponse
```

so FastAPI validates the output before returning it.

---

# 17. Basic Queue Request

Request:

```http
GET /api/v1/reviews/queue
```

returns all pending review cases.

Only unresolved `REVIEW_REQUIRED` documents are included.

---

# 18. Priority Filter

Example:

```http
GET /api/v1/reviews/queue?priority=HIGH
```

Lowercase values are also accepted:

```http
GET /api/v1/reviews/queue?priority=high
```

and normalized to:

```text
HIGH
```

---

# 19. Document-Type Filter

Example:

```http
GET /api/v1/reviews/queue?document_type=id_card
```

Values such as:

```text
ID_CARD
```

are normalized to:

```text
id_card
```

---

# 20. Combined Filters

Example:

```http
GET /api/v1/reviews/queue?priority=MEDIUM&document_type=guard_license
```

returns only:

```text
MEDIUM priority
+
guard_license
+
REVIEW_REQUIRED
+
not already human reviewed
```

documents.

---

# 21. Invalid Filter Handling

Invalid priorities are rejected.

Example:

```http
GET /api/v1/reviews/queue?priority=CRITICAL
```

Response:

```text
HTTP 400
```

Valid priorities are:

```text
HIGH
MEDIUM
LOW
```

---

Invalid document types are also rejected.

Example:

```http
GET /api/v1/reviews/queue?document_type=passport
```

Response:

```text
HTTP 400
```

Valid document types are:

```text
guard_license
sia_badge
id_card
```

---

# 22. Repository Compilation Test

The updated repository file was checked using:

```powershell
python -m py_compile .\src\db\repositories.py
```

Result:

```text
PASS
```

This confirmed that the new repository code contained no Python syntax or import errors.

---

# 23. Query Service Compilation Test

The query service was checked using:

```powershell
python -m py_compile .\src\db\query_service.py
```

Result:

```text
PASS
```

---

# 24. API Schema Compilation Test

The updated Pydantic schema module was tested with:

```powershell
python -m py_compile .\src\api\schemas.py
```

Result:

```text
PASS
```

---

# 25. FastAPI Compilation Test

The updated FastAPI application was tested with:

```powershell
python -m py_compile .\src\api\main.py
```

Result:

```text
PASS
```

---

# 26. PostgreSQL Review Queue Test

A real PostgreSQL integration test was implemented:

```text
test_phase7a_review_queue_isolated.py
```

Temporary database records were created for:

```text
HIGH REVIEW_REQUIRED
MEDIUM REVIEW_REQUIRED
MEDIUM REVIEW_REQUIRED
LOW REVIEW_REQUIRED
AUTO_ACCEPT
already human-reviewed REVIEW_REQUIRED
```

---

# 27. Existing Database Data Consideration

The first queue test initially expected exactly four records.

However, PostgreSQL already contained an existing pending document:

```text
guard_license.jpg
```

with:

```text
priority = MEDIUM
```

Therefore, the queue correctly returned:

```text
test data
+
existing real pending database data
```

This was not a repository error.

The test was improved so that it only asserted behaviour for temporary test document IDs while allowing legitimate existing database records to remain in the queue.

This made the test isolated and reliable without deleting real project data.

---

# 28. PostgreSQL Queue Test Results

Final isolated test results:

```text
[PASS] REVIEW_REQUIRED test documents returned
[PASS] AUTO_ACCEPT excluded
[PASS] Human-reviewed document excluded
[PASS] Priority ordering HIGH → MEDIUM → LOW
[PASS] Oldest-first ordering within same priority
[PASS] Priority filter
[PASS] Document-type filter
[PASS] Combined priority + document-type filter
[PASS] Empty filtered Phase 7A test result
```

It also correctly identified the existing pending record:

```text
Existing non-test pending documents in database: 1
  - guard_license.jpg (MEDIUM)
```

Final result:

```text
[PASS] PHASE 7A REVIEW QUEUE DATABASE TEST PASSED
```

---

# 29. FastAPI Review Queue Test

A dedicated API integration test was created:

```text
test_phase7a_review_queue_api.py
```

This tested the review queue through FastAPI rather than directly calling the query service.

---

# 30. API Test Coverage

The API test verified:

```text
GET /api/v1/reviews/queue
```

and confirmed:

* HTTP 200 response
* Pydantic response structure
* Pending review cases returned
* Correct priority ordering
* AUTO_ACCEPT exclusion
* Human-reviewed exclusion
* Priority filters
* Priority normalization
* Document-type filters
* Document-type normalization
* Combined filters
* Invalid priority rejection
* Invalid document-type rejection
* Empty valid filtered responses

---

# 31. FastAPI Test Results

Final output included:

```text
[PASS] GET /api/v1/reviews/queue returns HTTP 200
[PASS] Response schema contains required fields
[PASS] REVIEW_REQUIRED documents returned
[PASS] HIGH → MEDIUM → LOW ordering returned by API
[PASS] AUTO_ACCEPT excluded
[PASS] Human-reviewed document excluded
[PASS] Priority filter
[PASS] Priority normalization
[PASS] Document-type filter
[PASS] Document-type normalization
[PASS] Combined filters
[PASS] Invalid priority rejected with HTTP 400
[PASS] Invalid document type rejected with HTTP 400
[PASS] Empty valid filtered result handled correctly
```

Final result:

```text
[PASS] PHASE 7A REVIEW QUEUE FASTAPI TEST PASSED
```

---

# 32. Final Operational End-to-End Test

The final Phase 7A test was:

```text
test_phase7a_final_e2e.py
```

This tested the complete operational lifecycle rather than individual components.

The workflow was:

```text
Persist machine-processed document
        ↓
Machine analysis persisted
        ↓
Machine audit persisted
        ↓
Document appears in review queue
        ↓
Human APPROVE submitted through API
        ↓
Human review persisted
        ↓
Human audit persisted
        ↓
Document disappears from pending queue
        ↓
Stored machine result remains unchanged
        ↓
Audit history contains machine + human events
```

---

# 33. Machine Persistence Verification

A synthetic machine result was persisted using the real:

```text
PersistenceService
```

The test verified creation of:

```text
document_id
analysis_id
machine_audit_id
```

Result:

```text
[PASS] Machine-processed document persisted
[PASS] Machine analysis persisted
[PASS] Machine audit event persisted
```

---

# 34. Pending Queue Verification

After machine persistence, the endpoint:

```http
GET /api/v1/reviews/queue
```

was called.

The test confirmed that the new document appeared with:

```text
review_decision = REVIEW_REQUIRED
review_priority = MEDIUM
reason_codes = DOCUMENT_EXPIRED
```

Result:

```text
[PASS] Pending document appears in review queue
[PASS] Queue exposes trusted machine decision
```

---

# 35. Human Review Submission

A human approval was submitted through:

```http
POST /api/v1/documents/{document_id}/reviews
```

Request:

```json
{
  "reviewer_id": "phase7a-final-reviewer",
  "action": "APPROVE",
  "notes": "Phase 7A final end-to-end approval.",
  "corrections": null
}
```

Result:

```text
[PASS] Human APPROVE action submitted through API
[PASS] Human review persisted
[PASS] Human audit event persisted
```

---

# 36. Queue Removal Verification

The review queue was queried again after the human review.

The document was no longer present.

Result:

```text
[PASS] Human-reviewed document removed from pending queue
```

This confirms the core Phase 7A operational rule:

```text
Pending human review
        ↓
Human action submitted
        ↓
No longer pending
```

---

# 37. Machine Result Preservation

After human approval, the stored document was retrieved again.

The original machine review decision remained:

```text
REVIEW_REQUIRED
```

It was not overwritten with the human action.

Result:

```text
[PASS] Stored document remains available
[PASS] Original machine decision preserved
```

This maintains:

* Machine provenance
* Human provenance
* Auditability
* Historical traceability

---

# 38. Audit History Verification

The document history endpoint:

```http
GET /api/v1/documents/{document_id}/history
```

was tested.

The audit history contained:

```text
MACHINE_REVIEW_DECISION
HUMAN_REVIEW
```

Result:

```text
[PASS] Machine audit present in history
[PASS] Human review audit present in history
```

---

# 39. Direct PostgreSQL Verification

The final test also queried PostgreSQL directly to confirm that the document remained stored after the review action.

Result:

```text
[PASS] PostgreSQL document record verified
```

---

# 40. Final End-to-End Test Result

The final test completed successfully:

```text
========================================================================
[PASS] PHASE 7A FINAL END-TO-END TEST PASSED
========================================================================
```

Temporary test data was then removed:

```text
[CLEANUP] Phase 7A final test document removed.
```

---

# 41. Final Review Queue Architecture

The completed architecture is:

```text
                     PostgreSQL
                         │
            ┌────────────┴────────────┐
            │                         │
      documents                document_analyses
                                      │
                                      │ JSONB
                                      ▼
                              review_decision
                                      │
                                      ▼
                          ReviewQueueRepository
                                      │
                      ┌───────────────┴───────────────┐
                      │                               │
               decision filter                human review check
                      │                               │
             REVIEW_REQUIRED                 no existing review
                      │                               │
                      └───────────────┬───────────────┘
                                      │
                                      ▼
                           DocumentQueryService
                                      │
                                      ▼
                       GET /api/v1/reviews/queue
                                      │
                                      ▼
                               Human Reviewer
                                      │
                         APPROVE / REJECT / CORRECT
                                      │
                                      ▼
                                human_reviews
                                      │
                                      ▼
                                 audit_events
                                      │
                                      ▼
                         removed from pending queue
```

---

# 42. Review Queue Rules

The final operational rules are:

| Condition                           | Queue Status |
| ----------------------------------- | ------------ |
| `REVIEW_REQUIRED` + no human review | Included     |
| `AUTO_ACCEPT`                       | Excluded     |
| `REVIEW_REQUIRED` + APPROVE         | Excluded     |
| `REVIEW_REQUIRED` + REJECT          | Excluded     |
| `REVIEW_REQUIRED` + CORRECT         | Excluded     |

---

# 43. Review Priority Rules

Queue ordering:

| Priority | Queue Order |
| -------- | ----------: |
| HIGH     |           1 |
| MEDIUM   |           2 |
| LOW      |           3 |

Within each priority:

```text
oldest document first
```

---

# 44. Phase 7A Files Modified

The following existing files were updated:

```text
src/db/repositories.py
src/db/query_service.py
src/api/schemas.py
src/api/main.py
```

---

# 45. Phase 7A Test Files

The following test files were created:

```text
test_phase7a_review_queue.py
test_phase7a_review_queue_isolated.py
test_phase7a_review_queue_api.py
test_phase7a_final_e2e.py
```

The isolated version replaced the original test assumption that the database would contain no pre-existing pending review records.

---

# 46. Phase 7A Delivered Features

## Repository Layer

* [x] `ReviewQueueRepository`
* [x] JSONB machine-decision filtering
* [x] Human-review exclusion
* [x] Priority filtering
* [x] Document-type filtering
* [x] Priority ordering
* [x] Oldest-first ordering

## Query Service

* [x] Review queue service method
* [x] Filter normalization
* [x] API-ready serialization
* [x] Reason-code exposure
* [x] Review/anomaly issue exposure

## API Schemas

* [x] `ReviewQueueFilters`
* [x] `ReviewQueueItem`
* [x] `ReviewQueueResponse`
* [x] Strict review priority values
* [x] Typed timestamps

## FastAPI

* [x] `GET /api/v1/reviews/queue`
* [x] Priority query parameter
* [x] Document-type query parameter
* [x] Filter normalization
* [x] Invalid priority handling
* [x] Invalid document-type handling
* [x] Pydantic response validation

## Operational Workflow

* [x] Pending review enters queue
* [x] AUTO_ACCEPT stays out of queue
* [x] Human-reviewed cases leave queue
* [x] Machine result remains immutable
* [x] Human review remains separately persisted
* [x] Machine and human audit history preserved

---

# 47. Phase 7A Test Summary

| Test                                | Result |
| ----------------------------------- | ------ |
| Repository compile                  | ✅ PASS |
| Query service compile               | ✅ PASS |
| API schema compile                  | ✅ PASS |
| FastAPI compile                     | ✅ PASS |
| Real PostgreSQL queue test          | ✅ PASS |
| AUTO_ACCEPT exclusion               | ✅ PASS |
| Human-reviewed exclusion            | ✅ PASS |
| Priority ordering                   | ✅ PASS |
| Same-priority oldest-first ordering | ✅ PASS |
| Priority filter                     | ✅ PASS |
| Document-type filter                | ✅ PASS |
| Combined filters                    | ✅ PASS |
| Invalid priority HTTP 400           | ✅ PASS |
| Invalid document type HTTP 400      | ✅ PASS |
| FastAPI response validation         | ✅ PASS |
| Pending → review → removed workflow | ✅ PASS |
| Human review persistence            | ✅ PASS |
| Human audit persistence             | ✅ PASS |
| Machine result preservation         | ✅ PASS |
| Audit-history verification          | ✅ PASS |
| Direct PostgreSQL verification      | ✅ PASS |

---

# 48. Important Design Decision

The queue currently treats the existence of **any human review** as resolution of the pending case.

Therefore:

```text
first human action
→ remove from pending queue
```

This is suitable for the current workflow where:

```text
APPROVE
REJECT
CORRECT
```

are all considered terminal review actions.

If future requirements introduce:

```text
ASSIGNED
IN_REVIEW
ESCALATED
REOPENED
SECOND_REVIEW_REQUIRED
```

then a dedicated review-status state machine may be required.

---

# 49. Current Limitation

The review queue currently returns document metadata and machine-analysis information, but it does not provide the original uploaded document image.

The existing analysis workflow processes the image through a temporary file and deletes it after processing.

Therefore, although Phase 7A provides the backend review queue, a reviewer dashboard cannot yet display the original source document.

This becomes the next architectural requirement.

---

# 50. Next Phase

The next phase is:

```text
Phase 7B — Review Dashboard and Original Document Access
```

Before building the dashboard, the system should introduce persistent storage for the original uploaded document.

The intended future workflow is:

```text
Document Upload
      ↓
Permanent Original Document Storage
      ↓
Machine Processing
      ↓
REVIEW_REQUIRED
      ↓
Review Queue
      ↓
Reviewer Opens Case
      ↓
Original Document + Machine Extraction
      ↓
APPROVE / REJECT / CORRECT
```

---

# 51. Phase 7A Final Status

```text
Phase 7A — Review Queue Backend

Repository Layer                ✅ COMPLETE
Query Service                   ✅ COMPLETE
PostgreSQL Review Queue         ✅ COMPLETE
API Response Schemas            ✅ COMPLETE
FastAPI Review Queue Endpoint   ✅ COMPLETE
Filter Validation               ✅ COMPLETE
API Integration Tests           ✅ COMPLETE
Operational E2E Review Test     ✅ COMPLETE
Audit Verification              ✅ COMPLETE
Cleanup Verification            ✅ COMPLETE
```

# Phase 7A — COMPLETE ✅

Phase 7A successfully introduced a production-oriented pending-review queue backed by PostgreSQL and exposed through FastAPI.

The system can now automatically identify unresolved `REVIEW_REQUIRED` documents, prioritize them, filter them, expose them to a reviewer-facing application, and remove them from the pending queue once a human decision has been recorded.

This establishes the backend foundation required for the reviewer dashboard in the next phase.

```
